# Co-simulation start-time synchronization

Two or more DPsim processes running in real time need to start their loops at the same wall-clock instant. `InterfaceCosimSync` handles that rendezvous over TCP: one leader listens, every follower connects, and the leader broadcasts an absolute start time together with the timestep and duration. The blocking accept and connect are the synchronization, so there is no shared memory and no polling, and the peers may sit on different hosts.

This example runs one leader and two followers as separate processes on the local machine and shows that all of them agree on the same start instant.

In [ ]:
import multiprocessing as mp
import time

import dpsimpy

host = "127.0.0.1"
port = 47129

The follower connects to the leader, receives the configuration, and reports the start time it was handed. The timeout keeps it from waiting forever if no leader appears.

In [ ]:
def run_follower(index, result_queue):
    follower = dpsimpy.InterfaceCosimSync(f"follower{index}", host, port, "follower")
    follower.open()
    ok, start_time_ns, time_step_ns, duration_ns = follower.wait_for_config(10000)
    follower.close()
    result_queue.put((index, ok, start_time_ns, time_step_ns, duration_ns))

The leader opens its listening socket first, then both followers are started. The leader waits for both to connect, then sends a start time one second in the future, a one millisecond timestep and a one second duration. `expected_followers=2` makes the leader block until both have received and acknowledged the config, so it reports success only once the whole group is ready.

In [ ]:
ctx = mp.get_context("fork")
result_queue = ctx.Queue()

leader = dpsimpy.InterfaceCosimSync("leader", "", port, "leader")
leader.open()

followers = [ctx.Process(target=run_follower, args=(i, result_queue)) for i in range(2)]
for process in followers:
    process.start()

start_time_ns = time.time_ns() + 1_000_000_000
time_step_ns = 1_000_000
duration_ns = 1_000_000_000
published = leader.publish_config(
    start_time_ns, time_step_ns, duration_ns, expected_followers=2, timeout_ms=10000
)
leader.close()

results = sorted(result_queue.get() for _ in followers)
for process in followers:
    process.join()

Both followers received the exact start time the leader published, to the nanosecond. In a real setup each side would now pass that instant to `RealTimeSimulation.run(start_at)` so their loops step in lockstep.

In [ ]:
leader_start_ns = start_time_ns
print(f"leader published: {published}, start_time_ns: {leader_start_ns}")
for index, ok, start_time_ns, step_ns, dur_ns in results:
    print(
        f"follower{index}: ok={ok} start_time_ns={start_time_ns} time_step_ns={step_ns} duration_ns={dur_ns}"
    )

follower_starts = {start_time_ns for _, _, start_time_ns, _, _ in results}
assert published
assert all(ok for _, ok, _, _, _ in results)
assert follower_starts == {leader_start_ns}
print("all processes agree on the start instant")